#### **关于FDTD中mode expansion monitor的使用方法**

mode expansion monitor，即模式扩展监视器，目的是analyze the fraction of power transmitted into any mode(s) of a non-absorbing waveguide。模式展开的概念源于“模式分解”的思想，即在波导中可以将任意的电磁场解分解为一系列**本征模式**（正向、反向）的线性组合。

Ansys官网关于mode expansion monitor的描述：https://optics.ansys.com/hc/en-us/articles/360034902433-Using-and-understanding-Mode-Expansion-Monitors

##### 一、数学表达
假设波导支持正向的模式 $\varphi^{forward}_m$，其电场和磁场分别记为$\vec{E}_m^{forward}$ 和 $\vec{H}_m^{forward}$，对于反向的模式$\varphi^{backward}_m$同理，那么对于某个输入场，可以写为：
$$\vec{E}_{in}=\sum_{m}(a_m \vec{E}_m^{forward}+b_m \vec{E}_m^{backward})$$
$$\vec{H}_{in}=\sum_{m}(a_m \vec{H}_m^{forward}+b_m \vec{H}_m^{backward})$$
其中$a_m$和$b_m$分别为正向、反向传播分量的系数。

基于无吸收波导中传输模式的正交性，即：
$$\left \langle \varphi_m|\varphi_n\right \rangle=\frac{1}{2} \int\vec{E}_m \times\vec{H}_n^{*} \cdot d\vec{S}=N_m\delta_{mn}$$
其中$N_m$为归一化常数，所以有：

$$N_m = \frac{1}{2} \int \vec{E}_m \times \vec{H}_m^*\cdot d\vec{S}$$

于是有：
$$
\begin{aligned}
\frac{1}{2} \int\vec{E}_{in} \times\vec{H}_m^{*} \cdot d\vec{S}
& = \frac{1}{2} \int \sum_{m}(a_m \vec{E}_m^{forward}+b_m \vec{E}_m^{backward}) \times \vec{H}_m^{*} \cdot d\vec{S} \\
& = (a_m +b_m )N_m
\end{aligned}
$$
同理，有：
$$\frac{1}{2} \int\vec{E}_m^* \times\vec{H}_{in} \cdot d\vec{S}= (a_m - b_m )N_m^*$$

上述两式联立得：
$$
\left\{
\begin{array}{c}
    a_m = \frac14 (\frac{\int\vec{E}_{in}\times\vec{H}_m^{*}\cdot d\vec{S}}{N_m}+\frac{\int\vec{E}_{m}^*\times\vec{H}_{in}\cdot d\vec{S}}{N_m^*}) \\
    b_m = \frac14 (\frac{\int\vec{E}_{in}\times\vec{H}_m^{*}\cdot d\vec{S}}{N_m}-\frac{\int\vec{E}_{m}^*\times\vec{H}_{in}\cdot d\vec{S}}{N_m^*})
\end{array}
\right.
$$

再定义输入的总功率为：
$$P_{in} = \frac12 \int \vec{E}_{in} \times \vec{H}^*_{in} \cdot d\vec{S}$$

则可以把$a_m$和$b_m$转换成“这个输入场有多少功率（比例）在第m个模式中正向/反向传播”的表达。

##### 二、设置方法
1.模式扩展监视器包含两个部分，“mode calculation”部分需要用户选择一种或多种模式（即上面的$E_m$或$H_m$）来扩展输入的光场分布；“monitors for expansion”部分需要用户选择对应的monitor。

2.<mark>模式扩展监视器必须与由它的监视器放置在相同（大小）的横截面上。</mark>

##### 三、结果说明
- **T_total**:由光源注入功率归一化后的整体透射率，包括所有模式。
- **T_forward**:对于所选的mode(s),由光源注入功率归一化后的正向（坐标轴的正方向）透射率。
- **T_backward**:对于所选的mode(s),由光源注入功率归一化后的反向（坐标轴的负方向）透射率。
- **T_net**:对于所选的mode(s),由光源注入功率归一化后的净透射率，数值上应该等于T_forward - T_bakcward。
- **a,b**:对于所选的mode(s)，正向/反向的复传播系数，可以知道模态耦合的相位（延迟、相移等）信息。
- **N**:波导中模式的功率，先基于峰值将归一化使得$|E|^2=1$，再根据所在处截面的尺寸计算功率，因此这里会出现$N>P$的情况。
- **P**:监视器记录的输入总功率，以W为单位。

##### 四、案例
1.expansion1.fsp
<p align="center">
  <img src="resources/1.png" width="60%">
</p>

相同宽度的波导，光源沿x轴正方向注入，经过测试，在仿真区域内，无论将mode expansion monitor放置在哪里，甚至放置在monitor-R之前（x轴负方向），显示的一些结果均不变，即[T_total, T_forward, T_backward, T_net,N,P]=[1,1,0,1,2.1e-9,2.1e-9]。

2.expansion2.fsp
<p align="center">
  <img src="resources/2.png" width="60%">
</p>

光源由宽度为w1的窄波导中的基模传输到宽度为w2的宽波导中，由于上面N所描述的问题，如果直接计算T_forward其实会得到错误的结果，正确公式应为：$T_{forward}=\frac{|a_{out}|^2N_{out}}{|a_{in}|^2N_{in}}=\frac{|a_{out}|^2N_{out}}{sourcepower}$，在FDTD求解器中运行脚本计算发现两者值相同，说明mode expansion monitor已经正确的计算了透射率。

3.fa-pwb-taper.fsp
<p align="center">
  <img src="resources/3.png" width="60%">
</p>
这是自己设计的仿真PWB波导中连接光纤的锥形区域，使用import光源，将两个mode expansion monitor均放置在光纤段，对应到同一个monitor，发现改变其位置后T_forward的值发生了变化？

#### **关于FDTD中Ports的使用**

Ports可以充当模式源、监视器和模式扩展监视器的组合。

Ansys官网关于Ports的描述：https://optics.ansys.com/hc/en-us/articles/360034382554-Ports-FDTD-Simulation-Object

##### 一、一些基本设置
1.必须存在FDTD仿真区域才能添加Port；添加新的端口对象时，默认沿正x方向输入，且其余两个方向的span与FDTD仿真区域相同，可手动设置几何尺寸。

2.<mark>设置为光源的端口的注入方向用紫色箭头表示，其余端口的方向均以绿色箭头表示</mark>。

3.作为光源的端口同样可以选择mode；扫描波长等Global settings需要在Ports中设置；当source port中选择了多种模式时，在Ports中也可以对应选择不同的mode。

##### 二、用于模式扩展（以案例为例）
<p align="center">
  <img src="resources/4.png" width="60%">
</p>

左上角为输入源port 1，右上角为port 3，2，3，4均不注入。

- **T_in**:沿注入方向的透射率，当该port的方向设置为“forward”时，等效mode expansion monitor的T_forward。
- **T_out**:沿与注入相反方向的透射率，当该port的方向设置为“forward”时，等效mode expansion monitor的T_backward。
- **S**:可以表示为$$\frac{a_{monitor}*\sqrt{N_{monitor}}}{a_{source}*\sqrt{N_{source}}}$$

其余参数均与mode expansion monitor中相同。

##### 三、与mode expansion monitor对比
<p align="center">
  <img src="resources/5.png" width="60%">
</p>

<p align="center">
  <img src="resources/6.png" width="60%">
</p>
经过测试，一个使用mode source+field momitor+mode expansion monitor，效果与另外一个使用ports的结果几乎完全相同。


#### 关于FDTD中横截面监视器中光场出现的震荡现象

##### 一、问题描述
<p align="center">
  <img src="resources/7.png" width="100%">
</p>

当材料折射率发生突变时，光场会出现如上面的震荡现象，左图为仅LD区域，E非常平滑，右图的0-10um处为空气，整体的E就出现了震荡现象。经过mode expansion monitor测试（将mode expansion monitor对应到光源前的一个monitor上），无空气时几乎没有反射出现，而有空气时T_backward达到了21%的程度，说明反射不可忽略。

##### 二、问题解决
薄膜光学ppt链接：[薄膜光学](file:///C:/Users/22122/Desktop/学习/研一课/薄膜光学技术_第三章第一节.pptx)

以LD-PWB为例，对于单个波长$\lambda$，设LD的有效折射率为$n_1$，PWB的有效折射率为$n_2$，则薄膜的折射率由以下公式确定：
$$n_{thinfilm}=\sqrt{n_1 * n_2}$$
同时，薄膜的厚度满足以下公式：
$$d=\frac{\lambda/4}{n_{thinfilm}}$$

##### 三、案例
<p align="center">
  <img src="resources/8.png" width="60%">
</p>

加入AR膜之后，在1550nm处，反射回LD结构内的T_backward由10%降为1%，最后的T_forward达到了85%左右。

#### 关于FDTD与EME的对比
EME(Eigenmode Expansion)即特征模展开方法和FDTD方法是光学仿真中两种重要的数值方法，下面对比两者的优缺点（参考：https://www.youtube.com/watch?v=WgnuLuEFdXE）：

##### 1.算法概览和核心优势：

(1)FDTD：最通用的、在任意几何条件下、在具有复杂几何形状的纳米级器件中精确模拟光传播的方法。

原理：在时间轴上完整地求解麦克斯韦方程组，为时域方法。

核心优势：最为通用；并行计算能力强；宽带模拟，一次模拟即可获得某个光谱范围内的结果。

挑战：随着器件长度的增加，计算量增大，仿真时间随着网格精细化程度呈指数级增长（$1/ dx^4$）；对于较大的结构需要更多网格单元，网格色散的影响也可能更明显。

解决方法：灵活地使用mesh去针对关键区域进行更细的划分而不应用至整个器件；高度优化的计算引擎和计算策略。

(2)EME：一种常用的模拟波导或光纤中长距离传输的方法。

原理：将结构切片，计算每片的模式，并通过S矩阵/T矩阵进行双向传播。

核心优势：（长度无关性）仿真时间不随器件长度增加而增加，非常适合长距离波导，因此也可以非常快速地对器件的长度进行扫描；同时，非常适合仿真具有周期性的结构。

例如，将下面结构的长度由10um改为100um，并不会增加仿真时间。

<p align="center">
  <img src="resources/9.png" width="100%">
</p>

事实上，切分单元格的数量不由器件长度决定，而是由几何形状沿传播方向的变化大小决定，例如下图右侧的MMI多模干涉耦合器，可以用一个cell表示中间的大型且均匀结构。

<p align="center">
  <img src="resources/10.png" width="100%">
</p>

挑战：是频域方法，每个波长需独立模拟。

##### 2.应用场景对比
在聚合物波导与硅锥形波导的耦合中，耦合长度通常是一个需要考量的对象。对于下面的结构，3D FDTD仿真11个点通常需要花费超过6小时，而EME仿真101个点仅仅需要不超过2分钟。使用EME方法，无需重新计算模态就可以快速扫描锥形长度，且对于大结构长度，比FDTD更快。
<p align="center">
  <img src="resources/11.png" width="80%">
</p>
下面这个例子需要找到实现TE与TM模式转换的最佳锥形区长度，在这个例子中，EME不仅更快，且表现出更小的网格色散，当网格划分的更细时，FDTD才能获得与EME类似的结果。
<p align="center">
  <img src="resources/12.png" width="80%">
</p>
对于有波长扫描需求的场景，FDTD则胜出，例如下面的Y-branch分束器中，对于100个波长的仿真，FDTD所需时间则远小于EME。
<p align="center">
  <img src="resources/13.png" width="80%">
</p>
对于非平面结构，例如下面的3D光栅耦合器，EME方法则直接无法使用。
<p align="center">
  <img src="resources/14.png" width="80%">
</p>
对于周期性结构，例如下面的布拉格光栅波导，EME方法更加适合，且可以通过扫描器件的长度来获得传输波长，避免直接对波长进行扫描。
<p align="center">
  <img src="resources/15.png" width="80%">
</p>

##### 3.总结与建议
针对不同场景使用不同的方法。对于复杂结构，通常推荐先用EME进行快速的原型设计和长度扫描，最后使用FDTD进行最终的精度验证。